# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their `@id` as required by Croissant standards.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata
print(metadata.name + ": " + metadata.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities are referenced using their `@id`.

In [ ]:
# List all record sets by @id and view their fields

# Find all record sets in metadata
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = metadata.recordSet
else:
    # Some datasets attach record sets on metadata.hasPart
    try:
        record_sets = metadata.hasPart
    except AttributeError:
        pass

if not record_sets:
    print("No record sets found in metadata.")
else:
    # Print overview
    for rs in record_sets:
        print("RecordSet @id:", rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs)
        # List fields in each recordSet
        fields = []
        if isinstance(rs, dict) and 'field' in rs:
            fields = rs['field']
        elif hasattr(rs, 'field'):
            fields = rs.field
        if fields:
            print("  Fields:")
            for field in fields:
                if isinstance(field, dict):
                    print("    @id:", field.get('@id'))
                else:
                    print("    @id:", field)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

All entities must be referenced by their `@id`.

In [ ]:
# Extract data from each record set
record_set_ids = []
for rs in record_sets:
    if isinstance(rs, dict):
        record_set_ids.append(rs['@id'])
    else:
        record_set_ids.append(rs)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Print the columns and preview for each record set
for rsid in dataframes:
    print(f"RecordSet @id: {rsid}")
    print("Columns:", dataframes[rsid].columns.tolist())
    display(dataframes[rsid].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes, referencing columns by their `@id`.

In [ ]:
# Example analysis: select a record set with GAD-7 scores (anxiety), filter and normalize

# Try to identify a numeric field for demonstration, e.g., 'gad_7_total_score' or similar
example_record_set = None
numeric_field_id = None
group_field_id = None

for rsid, df in dataframes.items():
    for col in df.columns:
        # Try common score fields
        if 'gad_7' in col or 'phq_9' in col or 'psq' in col:
            example_record_set = rsid
            numeric_field_id = col
            break
    if example_record_set:
        break

# As a grouping field, try demographic fields
if example_record_set:
    df = dataframes[example_record_set]
    for col in df.columns:
        if 'gender' in col or 'age' in col or 'village' in col:
            group_field_id = col
            break

if example_record_set and numeric_field_id:
    print(f"Analyzing RecordSet: {example_record_set}, Numeric Field: {numeric_field_id}, Group Field: {group_field_id}")
    threshold = 10  # Example threshold for scores
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Grouping by demographic field if present
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No suitable numeric or grouping field found for EDA. Check column names in extracted DataFrames.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the example numeric field, grouped by group field if available
if example_record_set and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (RecordSet: {example_record_set})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable numeric or group fields for visualization found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was successfully loaded using its Croissant schema URL with `mlcroissant`.
- Record sets and their fields were listed with references to their `@id`.
- Data was extracted into pandas DataFrames, and key columns such as GAD-7 scores and demographic attributes were identified.
- EDA included filtering for high scores, normalization of numeric fields, and grouping by demographic fields (e.g., gender or village).
- Visualizations provided insights into the score distributions and their relationships with demographics.
